In [3]:
%store -r data_path

In [4]:

import json
from pathlib import Path
import pandas as pd

INPUT_DIR = Path(data_path)

files = [
    p for p in INPUT_DIR.rglob("*.json")
    if p.is_file()
]

records = []

for path in files:
    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            continue

        sections = data.get("sections", [])

        section_texts = [
            s.get("section_content", "")
            for s in sections
            if isinstance(s, dict)
        ]

        legal_text = " ".join(section_texts)

        records.append({
            "act_title": data.get("act_title", ""),
            "act_no": data.get("act_no", ""),
            "act_year": data.get("act_year", ""),
            "language": data.get("language", "unknown"),
            "legal_text": legal_text,
            "full_text": data.get("act_title", "") + " " + legal_text,
            "section_count": len(section_texts)
        })

    except Exception as e:
        print("Error:", path.name, e)

df = pd.DataFrame(records)

print("Total Acts:", len(df))
display(df.head())


Total Acts: 1484


,act_title,act_no,act_year,language,legal_text,full_text,section_count
0,"1The Districts Act, 1836",XXI,1836,english,"It shall be lawful for the2[***] Government, b...","1The Districts Act, 1836 It shall be lawful fo...",1
1,"The Societies Registration Act, 1860",XXI,1860,english,1. Any seven or more persons associated for an...,"The Societies Registration Act, 1860 1. Any se...",24
2,"The White Phosphorus Matches Prohibition Act, ...",V,1913,english,1. (1) This Act may be called theWhite Phospho...,"The White Phosphorus Matches Prohibition Act, ...",6
3,নির্দিষ্টকরণ (সম্পূরক) (২০০৬-০৭ অর্থ বৎসর) আইন...,১,2009,bengali,১। (১) এই আইননির্দিষ্টকরণ (সম্পূরক) (২০০৬-০৭ অ...,নির্দিষ্টকরণ (সম্পূরক) (২০০৬-০৭ অর্থ বৎসর) আইন...,4
4,"নির্দিষ্টকরণ (২০০৭-০৮ অর্থ বৎসর) আইন, ২০০৯",২,2009,bengali,১। (১) এই আইননির্দিষ্টকরণ (২০০৭-০৮ অর্থ বৎসর) ...,"নির্দিষ্টকরণ (২০০৭-০৮ অর্থ বৎসর) আইন, ২০০৯ ১। ...",4


In [5]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    max_features=100000,
    sublinear_tf=True
)

X = vectorizer.fit_transform(df["full_text"].fillna(""))

similarity_matrix = cosine_similarity(X)

print("Feature matrix:", X.shape)
print("Similarity matrix:", similarity_matrix.shape)


Feature matrix: (1484, 100000)
Similarity matrix: (1484, 1484)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


n = min(10, len(df))

plt.figure(figsize=(15, 12))

plt.imshow(
    similarity_matrix[:n, :n],
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(label="Cosine Similarity")

plt.title("Legal Acts - Similarity Heatmap")
plt.xlabel("act_title")
plt.ylabel("act_title")

plt.tight_layout()
plt.show()


In [12]:
for act_title in df["act_title"]:
    print(act_title)

1The Districts Act, 1836
The Societies Registration Act, 1860
The White Phosphorus Matches Prohibition Act, 1913
নির্দিষ্টকরণ (সম্পূরক) (২০০৬-০৭ অর্থ বৎসর) আইন, ২০০৯
নির্দিষ্টকরণ (২০০৭-০৮ অর্থ বৎসর) আইন, ২০০৯
নির্দিষ্টকরণ (সম্পূরক) (২০০৭-০৮ অর্থ বৎসর) আইন, ২০০৯
নির্দিষ্টকরণ (২০০৮-০৯ অর্থ বৎসর) আইন, ২০০৯
নির্বাচন কমিশন সচিবালয় আইন, ২০০৯
ভোটার তালিকা আইন, ২০০৯
মানিলণ্ডারিং প্রতিরোধ আইন, ২০০৯
অর্থ (২০০৭-২০০৮ অর্থ বৎসর) আইন, ২০০৯
অর্থ (২০০৮-২০০৯ অর্থ বৎসর) আইন, ২০০৯
সন্ত্রাস বিরোধী আইন, ২০০৯
The Mussalman Wakf Validating Act, 1913
ট্রেডমার্ক আইন, ২০০৯
তথ্য অধিকার আইন, ২০০৯
সিলেট মহানগরী পুলিশ আইন, ২০০৯
বরিশাল মহানগরী পুলিশ আইন, ২০০৯
ভোক্তা অধিকার সংরক্ষণ আইন, ২০০৯
বেগম রোকেয়া বিশ্ববিদ্যালয়, রংপুর আইন, ২০০৯
বাংলাদেশ ইউনিভার্সিটি অব প্রফেশনালস আইন, ২০০৯
পদ্মা বহুমুখী সেতু প্রকল্প (ভূমি অধিগ্রহণ) আইন, ২০০৯
নির্দিষ্টকরণ (সম্পূরক) আইন, ২০০৯
অর্থ আইন, ২০০৯
The Destructive Insects and Pests Act, 1914
নির্দিষ্টকরণ আইন, ২০০৯
সরকারি অর্থ ও বাজেট ব্যবস্থাপনা আইন, ২০০৯
বাংলাদেশ টেলিভিশনের জন্য টেরেস্ট